In [4]:
import torchvision
import torch
from torch import nn
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F

%matplotlib inline

In [5]:
mnist_train = torchvision.datasets.MNIST(
    "./mnist/", train=True, download=True, transform=torchvision.transforms.ToTensor()
)
mnist_val = torchvision.datasets.MNIST(
    "./mnist/", train=False, download=True, transform=torchvision.transforms.ToTensor()
)

train_dataloader = torch.utils.data.DataLoader(
    mnist_train, batch_size=4, shuffle=True, num_workers=1
)

val_dataloader = torch.utils.data.DataLoader(
    mnist_val, batch_size=4, shuffle=False, num_workers=1
)

In [6]:
def train(model: nn.Sequential, optimizer: torch.optim.Optimizer):
    for epoch in range(5):
        for x_train, y_train in tqdm(train_dataloader):
            y_pred = model(x_train)
            loss = F.cross_entropy(y_pred, y_train)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

def test(model: nn.Sequential):
    val_loss = []
    val_accuracy = []

    with torch.no_grad():
        for x_val, y_val in tqdm(val_dataloader):
            y_pred = model(x_val)
            loss = F.cross_entropy(y_pred, y_val)
            val_loss.append(loss.numpy())
            val_accuracy.extend(
                (torch.argmax(y_pred, dim=-1) == y_val).numpy().tolist()
                )

    return np.mean(val_loss), np.mean(val_accuracy)

In [ ]:
best_choice = {'lr': 0.001, 'weight_decay': 0.0001, 'n': 128}
best_res = 0.9767

lr = 0.001
w = 0.0001
n = 128

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, n),
    nn.ReLU(),
    nn.Linear(n, 10),
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=w)

train(model, optimizer)
loss, accuracy = test(model)

print(loss, accuracy)

if accuracy > best_res:
    print(f'{accuracy=}, {lr=}, {w=}, {n=}')
    best_res = accuracy
    best_choice['lr'] = lr
    best_choice['weight_decay'] = w
    best_choice['n'] = n

100%|██████████| 2500/2500 [00:10<00:00, 243.16it/s]

0.08173108 0.9776
